In [96]:
!pip install factor_analyzer

In [97]:
# importing all necessary libraries
import warnings              #importing library for turning off warnings which could clutter output
warnings.filterwarnings("ignore")

import os           #these used for handling files and maths
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler         #used for standardising data
from scipy import stats     #used for computing the trend slope
from factor_analyzer import (FactorAnalyzer,calculate_kmo,calculate_bartlett_sphericity)      #used for actual FA
                              
import matplotlib        #for plotting and saving
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe                   #used so plots are always readable
import itertools #used for leaderboard weightings



Some metrics were removed prior to analysis, such as Co2 in million tonnes which was replaced with Co2 per capita which made the analysis far more readable.
Renewable energy consumption was also removed due to the high negative correlation with electricity access, as low access to electicity can force higher renewable energy consumption.
Some economic metrics were also removed such as imports and final consumption expenditure, as these can correlate more with country size rather than development progress

In [98]:
os.makedirs("outputs", exist_ok=True)                #creating outputs folder to save everything to

EXCLUDE_TERRITORIES = {
    # Non-sovereign / dependent territories — not independent policy actors
    "Macao SAR, China", "Hong Kong SAR, China", "West Bank and Gaza",             #these countries aren't sovereign or they are dependent on other countries so won't be considered
    "Puerto Rico", "Channel Islands", "Isle of Man", "Faroe Islands",
    "New Caledonia", "French Polynesia", "Curacao", "Kosovo",
 
    "Iraq","Libya","Afghanistan", "Yemen",             # these are countries heavily conflict affected, so data is distorted by collapse so trajectory reflects recovery rather than devlopment
                
    "Venezuela, RB",                #heavy economic collapse causes misleading trajectory
    
    "Turkmenistan",                        #known to have innacurate stats as reported by governments
    "Belarus",
    
    "Maldives",                   #tiny island states with extreme metrics per capita due to tiny populations
    "Seychelles",
    "Cabo Verde",
}

#list of final metrics used after screening, with a sign applied to them based on if a higher value represents better development or not
METRICS = {
    #  Human development 
    "Life expectancy at birth, total (years) - SP.DYN.LE00.IN":                                     +1,
    "Prevalence of undernourishment (%) - SN_ITK_DEFC - 2.1.1":                                     -1,  
    "Proportion of population below international poverty line (%) - SI_POV_DAY1 - 1.1.1":          -1,  
    "Proportion of population using basic drinking water services (%) - SP_ACS_BSRVH2O - 1.4.1":    +1,
    "Access to electricity (% of population) - EG.ELC.ACCS.ZS":                                     +1,
    "Individuals using the Internet (% of population) - IT.NET.USER.ZS":                            +1,
    "Proportion of population covered by at least a 3G mobile network (%) - IT_MOB_3GNTWK - 9.c.1": +1,
    # Education 
    "Primary completion rate, total (% of relevant age group) - SE.PRM.CMPT.ZS":                    +1,
    "School enrollment, secondary (% gross) - SE.SEC.ENRR":                                         +1,
    "School enrollment, preprimary (% gross) - SE.PRE.ENRR":                                        +1,
    "Children out of school (% of primary school age) - SE.PRM.UNER.ZS":                            -1,  
    "Pupil-teacher ratio, primary - SE.PRM.ENRL.TC.ZS":                                             -1,  
    # Economy
    "GDP per capita (current US$) - NY.GDP.PCAP.CD":                                                +1,
    "Adjusted net national income per capita (annual % growth) - NY.ADJ.NNTY.PC.KD.ZG":            +1,
    "Gross savings (% of GDP) - NY.GNS.ICTR.ZS":                                                   +1,
    "Adjusted net savings, excluding particulate emission damage (% of GNI) - NY.ADJ.SVNX.GN.ZS":  +1,
    "Automated teller machines (ATMs) (per 100,000 adults) - FB.ATM.TOTL.P5":                      +1,
    "Cost of business start-up procedures, male (% of GNI per capita) - IC.REG.COST.PC.MA.ZS":     -1,  
    #  Governance & equity 
    "Women Business and the Law Index Score (scale 1-100) - SG.LAW.INDX":                          +1,
    "Proportion of seats held by women in national parliaments (%) - SG.GEN.PARL.ZS":              +1,
    # Natural resources / environment 
    "Total natural resources rents (% of GDP) - NY.GDP.TOTL.RT.ZS":                                -1,  
    "Adjusted savings: natural resources depletion (% of GNI) - NY.ADJ.DRES.GN.ZS":               -1,  
    "Adjusted savings: net forest depletion (% of GNI) - NY.ADJ.DFOR.GN.ZS":                      -1,  
    #  Green energy 
    "Renewable electricity output (% of total electricity output) - EG.ELC.RNEW.ZS":              +1,
    "CO2 per capita (tonnes)":                                                                      -1,  
    "Broad money (% of GDP) - FM.LBL.BMNY.GD.ZS":                                                 +1,
}

#These factor weights were a choice made based on how the factors correlate to development rather than current score. E.g wealth is downweighted to avoid richer countries just scoring higher
FACTOR_WEIGHTS = {
    "F1": 1.2,   # Education & Basic Development
    "F2": 1.0,   # Wealth & Financial Access 
    "F3": 1.2,   # Human & Social Development
    "F4": 1.3,   # Resource Management
    "F5": 1.3,   # Savings & Economic Sustainability
    "F6": 1.2,   # Governance & Gender Equity
}

#suitable number of factors to extract
N_FACTORS = 6

In [99]:
#loading dataset

print("STEP 1 — Load & prepare data")
df = pd.read_csv("WorldSustainabilityDataset.csv")

#changings co2 metric to avoid penalising larger countries
CO2_COL = "Annual production-based emissions of carbon dioxide (CO2), measured in million tonnes"
POP_COL = "Population, total - SP.POP.TOTL"
df["CO2 per capita (tonnes)"] = df[CO2_COL] * 1_000_000 / df[POP_COL]

#removing excluded countries
df = df[~df["Country Name"].isin(EXCLUDE_TERRITORIES)].copy()
print(f"  Countries after territory filter: {df['Country Name'].nunique()}")

#checking chosen metrics are within dataset
avail = [c for c in METRICS if c in df.columns]
missing_from_data = [c for c in METRICS if c not in df.columns]
if missing_from_data:
    print(f"  WARNING — not in dataset: {missing_from_data}")

#dropping countries missing more than 40% data
country_miss = df.groupby("Country Name")[avail].apply(lambda g: g.isna().mean().mean())
kept    = country_miss[country_miss <= 0.40].index
dropped = country_miss[country_miss >  0.40].index
df = df[df["Country Name"].isin(kept)].copy()
print(f"  Countries kept: {len(kept)}   dropped (>40% missing): {len(dropped)}")
if len(dropped):
    print(f"  Dropped: {', '.join(dropped)}")

#fill remaining gaps with median so there's no nans
for year, grp in df.groupby("Year"):
    df.loc[grp.index, avail] = grp[avail].fillna(grp[avail].median())
df[avail] = df[avail].fillna(df[avail].median())

#clipping extreme outliers to avoid individual stats distorting complete factor structure
for col in avail:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(lo, hi)
print(f"  Remaining NaN: {df[avail].isna().sum().sum()}")
print(f"  Final metric count: {len(avail)}")

#flipping sign of any metric as defined earlier
for col in avail:
    if METRICS[col] == -1:
        df[col] = -df[col]

flipped = [c for c in avail if METRICS[c] == -1]
print(f"\n  Sign-flipped {len(flipped)} metrics so higher = better:")
for c in flipped:
    print(f"    - {c[:70]}")

STEP 1 — Load & prepare data
  Countries after territory filter: 164
  Countries kept: 158   dropped (>40% missing): 6
  Dropped: Antigua and Barbuda, Aruba, Equatorial Guinea, Eritrea, Korea, Dem. People's Rep., Syrian Arab Republic
  Remaining NaN: 0
  Final metric count: 26

  Sign-flipped 9 metrics so higher = better:
    - Prevalence of undernourishment (%) - SN_ITK_DEFC - 2.1.1
    - Proportion of population below international poverty line (%) - SI_POV
    - Children out of school (% of primary school age) - SE.PRM.UNER.ZS
    - Pupil-teacher ratio, primary - SE.PRM.ENRL.TC.ZS
    - Cost of business start-up procedures, male (% of GNI per capita) - IC.
    - Total natural resources rents (% of GDP) - NY.GDP.TOTL.RT.ZS
    - Adjusted savings: natural resources depletion (% of GNI) - NY.ADJ.DRES
    - Adjusted savings: net forest depletion (% of GNI) - NY.ADJ.DFOR.GN.ZS
    - CO2 per capita (tonnes)


In [100]:
#taking mean for every country to give a good snapshot of countires profiles
country_means = df.groupby("Country Name")[avail].mean()
#standardise all metrics so numbers based metrics don't dominate over percentages
scaler = StandardScaler()
X = scaler.fit_transform(country_means)

In [101]:
#screening
#this section decides which metrics should be kept and should be dropped based on KMO and communality
ALL_CANDIDATES = list(METRICS.keys())
avail_all = [c for c in ALL_CANDIDATES if c in df.columns]

#running KMO on all variables, any with lower than 0.5 are removed as they don't share enough common variance to be studied
X_all = StandardScaler().fit_transform(
    df.groupby("Country Name")[avail_all].mean()
)
kmo_per_var_all, kmo_overall_all = calculate_kmo(X_all)

print("KMO on each metric:")
print(f"\n  Overall KMO (all candidates): {kmo_overall_all:.3f}\n")
print(f"  {'KMO':<8} {'Status':<20} Metric")


kmo_series = pd.Series(kmo_per_var_all, index=avail_all).sort_values()
for metric, val in kmo_series.items():
    if val < 0.5:
        status = "DROP — below 0.5"
    elif val < 0.6:
        status = "BORDERLINE"
    else:
        status = "OK"
    print(f"  {val:.3f}    {status:<20} {metric[:60]}")

#checking communality

print("Communalities after KMO")

X_screened = StandardScaler().fit_transform(
    df.groupby("Country Name")[avail].mean()
)
fa_diag = FactorAnalyzer(n_factors=N_FACTORS, rotation="varimax", method="ml")
fa_diag.fit(X_screened)
comm_diag = pd.Series(fa_diag.get_communalities(), index=avail).sort_values()

print(f"\n  {'Communality':<14} {'Status':<20} Metric")
for metric, val in comm_diag.items():
    if val < 0.30:
        status = "DROP — below 0.30"
    elif val < 0.50:
        status = "ACCEPTABLE"
    else:
        status = "GOOD"
    print(f"  {val:.3f}         {status:<20} {metric[:60]}")

print(f"\n  Metrics retained after both screens: {len(avail)}")
print(f"  Metrics dropped at KMO stage:        {len(avail_all) - len(avail)}")

#this leaves 26 metrics overall to analyse

KMO on each metric:

  Overall KMO (all candidates): 0.868

  KMO      Status               Metric
  0.543    BORDERLINE           Adjusted net national income per capita (annual % growth) - 
  0.576    BORDERLINE           Gross savings (% of GDP) - NY.GNS.ICTR.ZS
  0.587    BORDERLINE           Adjusted net savings, excluding particulate emission damage 
  0.632    OK                   Adjusted savings: natural resources depletion (% of GNI) - N
  0.710    OK                   Total natural resources rents (% of GDP) - NY.GDP.TOTL.RT.ZS
  0.751    OK                   Proportion of seats held by women in national parliaments (%
  0.761    OK                   Renewable electricity output (% of total electricity output)
  0.767    OK                   Women Business and the Law Index Score (scale 1-100) - SG.LA
  0.774    OK                   Adjusted savings: net forest depletion (% of GNI) - NY.ADJ.D
  0.842    OK                   Children out of school (% of primary school age) - 

In [102]:
#kmo and bartlett test

print("STEP 2 — Suitability tests")

kmo_per_var, kmo_overall = calculate_kmo(X)
chi2, p_bartlett = calculate_bartlett_sphericity(X)

print(f"\n  Kaiser-Meyer-Olkin (KMO) measure of sampling adequacy")
print(f"    Overall KMO     : {kmo_overall:.3f}  ", end="")
if kmo_overall >= 0.9:    print("(Marvellous)")
elif kmo_overall >= 0.8:  print("(Meritorious)")
elif kmo_overall >= 0.7:  print("(Middling)")
elif kmo_overall >= 0.6:  print("(Mediocre)")
else:                     print("(Unacceptable)")

print(f"\n  Per-variable KMO:")
kmo_df = pd.Series(kmo_per_var, index=avail).sort_values()
for col, val in kmo_df.items():
    tag = " [ACCEPTABLE]" if val >= 0.5 else " [*** BELOW THRESHOLD]"
    print(f"    {val:.3f}  {col[:65]}{tag}")

print(f"\n  Bartlett's Test of Sphericity")
print(f"    Chi-square : {chi2:.2f}")
print(f"    p-value    : {p_bartlett:.2e}  ", end="")
print("(Significant — correlations exist, FA is appropriate)" if p_bartlett < 0.05
      else "(NOT significant — FA may not be appropriate)")

STEP 2 — Suitability tests

  Kaiser-Meyer-Olkin (KMO) measure of sampling adequacy
    Overall KMO     : 0.868  (Meritorious)

  Per-variable KMO:
    0.543  Adjusted net national income per capita (annual % growth) - NY.AD [ACCEPTABLE]
    0.576  Gross savings (% of GDP) - NY.GNS.ICTR.ZS [ACCEPTABLE]
    0.587  Adjusted net savings, excluding particulate emission damage (% of [ACCEPTABLE]
    0.632  Adjusted savings: natural resources depletion (% of GNI) - NY.ADJ [ACCEPTABLE]
    0.710  Total natural resources rents (% of GDP) - NY.GDP.TOTL.RT.ZS [ACCEPTABLE]
    0.751  Proportion of seats held by women in national parliaments (%) - S [ACCEPTABLE]
    0.761  Renewable electricity output (% of total electricity output) - EG [ACCEPTABLE]
    0.767  Women Business and the Law Index Score (scale 1-100) - SG.LAW.IND [ACCEPTABLE]
    0.774  Adjusted savings: net forest depletion (% of GNI) - NY.ADJ.DFOR.G [ACCEPTABLE]
    0.842  Children out of school (% of primary school age) - SE.PRM.UN

In [103]:
#deciding how many factors to use using Kaiser and Scree plot

print("STEP 3 — Scree plot & eigenvalue analysis")

#fit full FA with all eign=envalues 
fa_full = FactorAnalyzer(n_factors=len(avail), rotation=None, method="ml")
fa_full.fit(X)
ev, _ = fa_full.get_eigenvalues()

#each factor should explain as much variance as a single variable. an eigenvalue of greater than 1 is doing useful work
n_kaiser = (ev > 1).sum()
print(f"\n  Eigenvalues: {[f'{e:.3f}' for e in ev[:12]]}")
print(f"  Kaiser criterion (eigenvalue > 1): {n_kaiser} factors")
print(f"  Chosen n_factors = {N_FACTORS}  (Kaiser criterion + interpretability)")

#scree plot plots each factor's eignenvalue. Factors to the right of the 'elbow' are dropped
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(ev) + 1), ev, "bo-", markersize=6, linewidth=1.8, label="Eigenvalue")


ax.axhline(y=1, color="red", linestyle="--", linewidth=1.2, label="Kaiser threshold (λ = 1)")


ax.axvline(x=N_FACTORS + 0.5, color="green", linestyle=":", linewidth=1.5,
           label=f"Chosen: {N_FACTORS} factors")
ax.fill_between(range(1, N_FACTORS + 1), ev[:N_FACTORS], alpha=0.12, color="green")

ax.set_xlabel("Factor Number", fontsize=11)
ax.set_ylabel("Eigenvalue", fontsize=11)
ax.set_title("Scree Plot\n"
             f"KMO = {kmo_overall:.3f} "
             f"{N_FACTORS} factors selected",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.set_xlim(0.5, min(len(ev), 20) + 0.5)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/scree_plot.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: scree_plot.png")

STEP 3 — Scree plot & eigenvalue analysis

  Eigenvalues: ['10.611', '2.705', '1.945', '1.573', '1.503', '1.099', '0.766', '0.755', '0.628', '0.613', '0.490', '0.463']
  Kaiser criterion (eigenvalue > 1): 6 factors
  Chosen n_factors = 6  (Kaiser criterion + interpretability)
  Saved: scree_plot.png


In [104]:
# now we run the actual factor analysis
# varimax rotation is used because it makes the factors easier to interpret —
# it rotates the factor axes so each variable loads highly on as few factors
# as possible, giving cleaner, more distinct factors
# ML (maximum likelihood) is the estimation method — more statistically rigorous
# than the default principal axis factoring
#running actual factor analysis

print("STEP 4 — Factor Analysis (varimax, ML)")

fa = FactorAnalyzer(n_factors=N_FACTORS, rotation="varimax", method="ml")
fa.fit(X)

# extract the key outputs from the fitted model
factor_cols   = [f"F{i+1}" for i in range(N_FACTORS)]
loadings_df   = pd.DataFrame(fa.loadings_, index=avail, columns=factor_cols)  # how strongly each metric relates to each factor
communalities = pd.Series(fa.get_communalities(), index=avail)                 # how much of each metric's variance is captured overall
var_stats     = fa.get_factor_variance()                                       # SS loadings, proportion of variance, cumulative

#showing how much each factor explains total variance
print(f"\n  Variance explained:")
print(f"  {'Factor':<8} {'SS Loadings':>12} {'Proportion':>12} {'Cumulative':>12}")
print("  " + "-" * 48)
for i, (ss, prop, cum) in enumerate(zip(*var_stats)):
    print(f"  F{i+1:<7} {ss:>12.3f} {prop*100:>11.1f}% {cum*100:>11.1f}%")
print(f"\n  Total variance explained: {var_stats[1].sum()*100:.1f}%")

#factor loading matrix shows how strong each metric relates to each factor 
print(f"\n  Factor Loading Matrix (showing |loading| > 0.40):\n")
THRESHOLD = 0.40
short_names = [m.split(" - ")[0][:52] for m in avail]
header = f"  {'Metric':<54}" + "".join(f"  {c:>6}" for c in factor_cols)
print(header)
print("  " + "-" * (54 + 10 * N_FACTORS))
for metric, short in zip(avail, short_names):
    row_vals = []
    for fc in factor_cols:
        val = loadings_df.loc[metric, fc]
        if abs(val) >= THRESHOLD:
            row_vals.append(f"{val:>+7.2f}")
        else:
            row_vals.append(f"{'':>7}")   # blank if below threshold, keeps table readable
    print(f"  {short:<54}" + "".join(row_vals))

#how much each metric variance is explained by 6 factors combined
print(f"\n  Communalities (proportion of variance explained per metric):\n")
for metric, val in communalities.sort_values().items():
    bar   = "█" * int(val * 20)    # visual bar chart in the console output
    flag  = "  [low]" if val < 0.30 else ""
    short = metric.split(" - ")[0][:55]
    print(f"  {val:.3f}  {bar:<20}  {short}{flag}")

#printing top 3 loading metrics to help decide the factor names
print(f"\n  Top 3 loadings per factor (for naming):\n")
factor_top = {}
for fc in factor_cols:
    top3 = loadings_df[fc].abs().nlargest(3).index.tolist()
    factor_top[fc] = top3
    print(f"  {fc}:")
    for m in top3:
        print(f"    {loadings_df.loc[m, fc]:+.3f}  {m[:70]}")
    print()

STEP 4 — Factor Analysis (varimax, ML)

  Variance explained:
  Factor    SS Loadings   Proportion   Cumulative
  ------------------------------------------------
  F1              4.293        16.5%        16.5%
  F2              4.197        16.1%        32.7%
  F3              3.253        12.5%        45.2%
  F4              2.245         8.6%        53.8%
  F5              1.877         7.2%        61.0%
  F6              1.758         6.8%        67.8%

  Total variance explained: 67.8%

  Factor Loading Matrix (showing |loading| > 0.40):

  Metric                                                      F1      F2      F3      F4      F5      F6
  ------------------------------------------------------------------------------------------------------------------
  Life expectancy at birth, total (years)                 +0.69  +0.42                            
  Prevalence of undernourishment (%)                      +0.68                                   
  Proportion of population b

In [105]:
#came up with appropiate factor names
FACTOR_NAMES = {
    "F1": "Education & Basic Dev.",
    "F2": "Wealth & Financial Access",
    "F3": "Human & Social Development",
    "F4": "Resource Management",
    "F5": "Savings & Econ. Sustainability",
    "F6": "Governance & Gender Equity",
}

print("STEP 5 — Computing factor scores (all country-years)")


#apply standardiser for years as well to help with trajectory
X_panel = scaler.transform(df[avail].values)

#uses factor lodings to each country-year position
factor_scores = fa.transform(X_panel)

scores_df = pd.DataFrame(factor_scores, columns=factor_cols, index=df.index)
scores_df["Country Name"] = df["Country Name"].values
scores_df["Year"]         = df["Year"].values


#combining 6 factor scores into a single comnpsite score using weights
w = np.array([FACTOR_WEIGHTS[fc] for fc in factor_cols], dtype=float)
w /= w.sum()
scores_df["Composite Score"] = factor_scores @ w

print(f"  Weights: { {fc: f'{wi:.2f}' for fc, wi in zip(factor_cols, w)} }")
print(f"  Scores computed for {len(scores_df)} country-year observations.")


STEP 5 — Computing factor scores (all country-years)
  Weights: {'F1': '0.17', 'F2': '0.14', 'F3': '0.17', 'F4': '0.18', 'F5': '0.18', 'F6': '0.17'}
  Scores computed for 3002 country-year observations.


In [106]:
#now building leaderboard 
print("STEP 6 — Investment leaderboard")


#weight investment score based on current score and trajectory 
WEIGHT_CURRENT    = 0.40
WEIGHT_TRAJECTORY = 0.60

#fitting straight line through compposite scores over time and returns slope
def ols_slope(series):
    if series.nunique() < 3:
        return 0.0
    yrs = np.arange(len(series))
    slope, _, _, pval, _ = stats.linregress(yrs, series.values)
    return slope if pval < 0.15 else 0.0

# calculate current score and trajectory for every country
records = []
for country, grp in scores_df.groupby("Country Name"):
    g = grp.sort_values("Year")
    records.append({
        "Country":       country,
        "Current Score": g.tail(3)["Composite Score"].mean(),   # average of last 3 years to smooth noise
        "Trajectory":    ols_slope(g.set_index("Year")["Composite Score"]),
        **{fc: g.iloc[-1][fc] for fc in factor_cols},           # most recent year's individual factor scores
    })

lb = pd.DataFrame(records)

#damping trajectory for countries in bottom 20% to stop countries which crashed and are recovering looking like great investments
p20 = lb["Current Score"].quantile(0.20)
lb.loc[lb["Current Score"] < p20, "Trajectory"] *= 0.5

#normalising series so trajectory and current score are on the same scale
def minmax(s):
    r = s.max() - s.min()
    return (s - s.min()) / r if r > 0 else pd.Series(0.5, index=s.index)

lb["Current Norm"]     = minmax(lb["Current Score"])
lb["Trajectory Norm"]  = minmax(lb["Trajectory"])

#final investment is a weighted blend
lb["Investment Score"] = WEIGHT_CURRENT * lb["Current Norm"] + WEIGHT_TRAJECTORY * lb["Trajectory Norm"]

lb = lb.sort_values("Investment Score", ascending=False).reset_index(drop=True)
lb["Rank"] = lb.index + 1

#put data like income classification and regime type in leaderboard
meta = (df[["Country Name", "Income Classification (World Bank Definition)",
            "Regime Type (RoW Measure Definition)", "Continent"]]
        .drop_duplicates("Country Name")
        .rename(columns={"Country Name": "Country"}))
lb = lb.merge(meta, on="Country", how="left")

# print the top 30 countries
print(f"\n  {'Rank':<5} {'Country':<30} {'Inv.Score':>10} {'Current':>9} "
      f"{'Traj':>8}  {'Income':<20} Regime")
print("  " + "-" * 100)
for _, row in lb.head(30).iterrows():
    income = str(row.get("Income Classification (World Bank Definition)", ""))[:18]
    regime = str(row.get("Regime Type (RoW Measure Definition)", ""))[:22]
    print(f"  {int(row['Rank']):<5} {row['Country']:<30} "
          f"{row['Investment Score']:>10.4f} "
          f"{row['Current Score']:>9.3f} "
          f"{row['Trajectory']:>8.4f}  {income:<20} {regime}")

#save outputs to csvs 
out_cols = (["Rank", "Country", "Investment Score", "Current Norm",
             "Trajectory Norm", "Current Score", "Trajectory",
             "Income Classification (World Bank Definition)",
             "Regime Type (RoW Measure Definition)", "Continent"]
            + factor_cols)
lb[out_cols].to_csv("outputs/investment_leaderboard.csv", index=False)
scores_df.to_csv("outputs/country_factor_scores.csv", index=False)
loadings_df.to_csv("outputs/factor_loadings.csv")
communalities.to_frame("Communality").to_csv("outputs/communalities.csv")

var_table = pd.DataFrame({
    "SS Loadings":    var_stats[0],
    "Proportion (%)": var_stats[1] * 100,
    "Cumulative (%)": var_stats[2] * 100,
}, index=factor_cols)
var_table.to_csv("outputs/factor_variance.csv")

print("\n  Saved all CSVs.")
    

STEP 6 — Investment leaderboard

  Rank  Country                         Inv.Score   Current     Traj  Income               Regime
  ----------------------------------------------------------------------------------------------------
  1     Qatar                              0.9005     0.596   0.1003  High income          Closed Autocracy
  2     Brunei Darussalam                  0.7015     0.615   0.0594  High income          nan
  3     Ireland                            0.6911     1.019   0.0421  High income          Liberal Democracy
  4     Singapore                          0.6692     1.130   0.0335  High income          Electoral Autocracy
  5     North Macedonia                    0.6506     0.623   0.0488  Lower-middle incom   Electoral Autocracy
  6     Korea, Rep.                        0.6327     0.974   0.0320  Upper-middle incom   Liberal Democracy
  7     Austria                            0.6216     0.838   0.0348  High income          Liberal Democracy
  8     Malta 

In [107]:

#seeing how top 10 ranking changes across all different weightings of current score and trajectory using spearman rank coefficients
weight_combinations = [
    (0.30, 0.70),
    (0.40, 0.60),
    (0.45, 0.55),   # our chosen weights
    (0.50, 0.50),
    (0.60, 0.40),
    (0.70, 0.30),
]

print(f"{'Weights (curr/traj)':<22}", end="")
for i in range(1, 11):
    print(f"  #{i:<3}", end="")
print()
print("-" * 120)

rankings = {}
for w_curr, w_traj in weight_combinations:
    lb_test = lb.copy()
    lb_test["Test Score"] = w_curr * lb_test["Current Norm"] + w_traj * lb_test["Trajectory Norm"]
    lb_test = lb_test.sort_values("Test Score", ascending=False).reset_index(drop=True)
    top10 = lb_test.head(10)["Country"].tolist()
    rankings[(w_curr, w_traj)] = top10
    
    label = f"{w_curr:.0%} / {w_traj:.0%}"
    marker = " ← chosen" if (w_curr, w_traj) == (0.45, 0.55) else ""
    print(f"{label:<22}{marker}", end="")
    for country in top10:
        print(f"  {country[:5]:<5}", end="")
    print()

print("\n  Spearman rank correlation with chosen weights (0.45/0.55):\n")
from scipy.stats import spearmanr

chosen_scores = (0.45 * lb["Current Norm"] + 0.55 * lb["Trajectory Norm"])
for w_curr, w_traj in weight_combinations:
    alt_scores = w_curr * lb["Current Norm"] + w_traj * lb["Trajectory Norm"]
    corr, pval = spearmanr(chosen_scores, alt_scores)
    label = f"{w_curr:.0%} / {w_traj:.0%}"
    print(f"  {label:<12}  r = {corr:.4f}  (p = {pval:.2e})")

Weights (curr/traj)     #1    #2    #3    #4    #5    #6    #7    #8    #9    #10 
------------------------------------------------------------------------------------------------------------------------
30% / 70%               Qatar  Brune  Irela  North  Uzbek  Singa  Malta  Alger  Ethio  Korea
40% / 60%               Qatar  Brune  Irela  Singa  North  Korea  Austr  Malta  Germa  Denma
45% / 55%              ← chosen  Qatar  Irela  Brune  Singa  North  Korea  Austr  Denma  Switz  Germa
50% / 50%               Qatar  Irela  Singa  Brune  Korea  North  Switz  Denma  Austr  Germa
60% / 40%               Qatar  Singa  Irela  Korea  Switz  Brune  Denma  Austr  Germa  Norwa
70% / 30%               Singa  Qatar  Irela  Switz  Korea  Denma  Norwa  Nethe  Austr  Germa

  Spearman rank correlation with chosen weights (0.45/0.55):

  30% / 70%     r = 0.9848  (p = 1.37e-120)
  40% / 60%     r = 0.9980  (p = 5.83e-189)
  45% / 55%     r = 1.0000  (p = 0.00e+00)
  50% / 50%     r = 0.9983  (p = 8.

In [110]:
#generating visualisations and saving as a png
print("STEP 7 — Generating charts")

fn_labels = [f"{fc}: {FACTOR_NAMES[fc]}" for fc in factor_cols]

#generating top 30 leaderboard
fig, ax = plt.subplots(figsize=(14, 11))
top30  = lb.head(30)
colors = plt.cm.RdYlGn(np.linspace(0.35, 0.95, len(top30)))
bars   = ax.barh(top30["Country"][::-1], top30["Investment Score"][::-1],
                 color=colors[::-1], edgecolor="white", linewidth=0.5)
for bar, score in zip(bars, top30["Investment Score"][::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f"{score:.3f}", va="center", ha="left", fontsize=7.5, color="#333")
ax.set_xlabel("Investment Score", fontsize=11)
ax.set_title(
    f"Top 30 Countries Leaderboard on Investment score\n",
    fontsize=11, fontweight="bold")
ax.set_xlim(0, top30["Investment Score"].max() * 1.13)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/top30_leaderboard.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: top30_leaderboard.png")


#factor loadings heatmap showing how strong each metric relates to the factors
fig, ax = plt.subplots(figsize=(12, 14))
short_metrics = [m.split(" - ")[0][:50] for m in loadings_df.index]
im = ax.imshow(loadings_df.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(N_FACTORS))
ax.set_yticks(range(len(short_metrics)))
ax.set_xticklabels([f"{fc}\n{FACTOR_NAMES[fc]}" for fc in factor_cols],
                   fontsize=8.5, rotation=10)
ax.set_yticklabels(short_metrics, fontsize=7)
for i in range(len(avail)):
    for j in range(N_FACTORS):
        val = loadings_df.iloc[i, j]
        tc  = "white" if abs(val) > 0.6 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6.5, color=tc)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
ax.set_title("Factor Loadings Heatmap  (varimax rotation)\n"
             "Red = high positive loading, Blue = high negative",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/factor_loadings_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: factor_loadings_heatmap.png")


#showing how well each metric is captured by 6 factors
fig, ax = plt.subplots(figsize=(12, 8))
comm_sorted = communalities.sort_values(ascending=True)
short_c = [m.split(" - ")[0][:52] for m in comm_sorted.index]
bar_colors = ["#e74c3c" if v < 0.30 else "#f39c12" if v < 0.50 else "#27ae60"
              for v in comm_sorted]
ax.barh(short_c, comm_sorted, color=bar_colors, edgecolor="white")
ax.axvline(0.30, color="#e74c3c", ls="--", lw=1.2, label="Min threshold (0.30)")
ax.axvline(0.50, color="#f39c12", ls="--", lw=1.2, label="Good threshold (0.50)")
ax.set_xlabel("Communality (proportion of variance explained)", fontsize=10)
ax.set_title(f"Communalities — {N_FACTORS}-Factor Solution\n"
             "How much of each metric's variance is captured by the 6 factors",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/communalities.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: communalities.png")

#current score against trajectory split into 4 quadrants
fig, ax = plt.subplots(figsize=(14, 10))
x, y   = lb["Current Norm"], lb["Trajectory Norm"]
xm, ym = x.median(), y.median()   # quadrant dividers set at median so each quadrant has similar number of countries

quad_cfg = {
    "High current and improving":       ("#27ae60", x >= xm,  y >= ym),
    "High current, stagnated":          ("#2980b9", x >= xm,  y <  ym),
    "Lower current, improving": ("#e67e22", x <  xm,  y >= ym),
    "Low current,minimal improvement":            ("#c0392b", x <  xm,  y <  ym),
}
for label, (color, mx, my) in quad_cfg.items():
    mask = mx & my
    ax.scatter(x[mask], y[mask], c=color, s=60, alpha=0.75, zorder=3,
               label=f"{label}  (n={mask.sum()})", edgecolors="white", linewidths=0.4)

top30_set = set(lb.head(30)["Country"])
for _, row in lb.iterrows():
    is_top = row["Country"] in top30_set
    ax.annotate(row["Country"],
                xy=(row["Current Norm"], row["Trajectory Norm"]),
                fontsize=6.5 if is_top else 5.2,
                fontweight="bold" if is_top else "normal",
                xytext=(3, 3), textcoords="offset points",
                color="#111" if is_top else "#555")

ax.axvline(xm, color="gray", lw=1.0, ls="--", alpha=0.5)
ax.axhline(ym, color="gray", lw=1.0, ls="--", alpha=0.5)
kw = dict(fontsize=10, fontweight="bold", alpha=0.15, zorder=1)
ax.text(xm+0.01, y.max()-0.02, "High current and improving",       color="#27ae60", va="top",    **kw)
ax.text(0.01,    y.max()-0.02, "Lower current, improving",  color="#e67e22", va="top",    **kw)
ax.text(xm+0.01, y.min()+0.02, "High current, stagnated",          color="#2980b9", va="bottom", **kw)
ax.text(0.01,    y.min()+0.02, "Low current,minimal improvement",             color="#c0392b", va="bottom", **kw)
ax.set_xlabel("Current Score (normalised)", fontsize=11)
ax.set_ylabel("Trajectory / Momentum (normalised)", fontsize=11)
ax.set_title("All Countries: Current Score vs Trajectory\n"
             "Top 30 countries labelled in bold", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/score_vs_trajectory.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: score_vs_trajectory.png")


#showing how well all metrics correlate with each other
country_means_raw = df.groupby("Country Name")[avail].mean()
metric_corr = country_means_raw.corr()
metric_corr.to_csv("outputs/metric_correlation_matrix.csv")

short_labels = [m.split(" - ")[0][:42] for m in metric_corr.columns]
fig, ax = plt.subplots(figsize=(18, 16))
im = ax.imshow(metric_corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=55, ha="right", fontsize=6.5)
ax.set_yticklabels(short_labels, fontsize=6.5)
plt.colorbar(im, ax=ax, fraction=0.015, pad=0.02)
ax.set_title("Metric Correlation Matrix  (country-level means)",
             fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("outputs/metric_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: metric_correlation_heatmap.png")

#showing how top countries score across all factors
top40 = lb.head(40)["Country"].tolist()
cf = (scores_df[scores_df["Country Name"].isin(top40)]
      .groupby("Country Name")[factor_cols]
      .mean()
      .reindex(top40))
cf_norm = (cf - cf.min()) / (cf.max() - cf.min())
short_fn = [f"{fc}: {FACTOR_NAMES[fc]}" for fc in factor_cols]

fig, ax = plt.subplots(figsize=(12, 14))
im = ax.imshow(cf_norm.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(N_FACTORS))
ax.set_yticks(range(len(top40)))
ax.set_xticklabels(short_fn, rotation=20, ha="right", fontsize=8.5)
ax.set_yticklabels(top40, fontsize=8)
for i, country in enumerate(top40):
    for j, fc in enumerate(factor_cols):
        raw = cf.loc[country, fc]
        nv  = cf_norm.loc[country, fc]
        tc  = "white" if nv < 0.25 or nv > 0.82 else "black"
        ax.text(j, i, f"{raw:.2f}", ha="center", va="center", fontsize=6.5, color=tc)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label="Normalised score (0–1)")
ax.set_title("Country × Factor Heatmap — Top 40 Countries",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/country_factor_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: country_factor_heatmap.png")

#for every factor showing how top countries perform for each factor over time
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()
cmap = plt.cm.tab10

for ax_idx, fc in enumerate(factor_cols):
    ax = axes[ax_idx]
    avg = scores_df.groupby("Country Name")[fc].mean().nlargest(8)
    for i, country in enumerate(avg.index):
        sub = scores_df[scores_df["Country Name"] == country].sort_values("Year")
        color = cmap(i / 8)
        ax.plot(sub["Year"], sub[fc], marker="o", markersize=3.5,
                linewidth=1.8, color=color, label=country)
        last = sub.iloc[-1]
        # label at the end of each line so we don't need a separate legend
        ax.annotate(country, xy=(last["Year"], last[fc]),
                    xytext=(4, 0), textcoords="offset points",
                    fontsize=6.5, color=color, va="center",
                    path_effects=[pe.withStroke(linewidth=2, foreground="white")])
    ax.set_title(f"{fc}: {FACTOR_NAMES[fc]}\n(top 8 by avg score)",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Year", fontsize=9)
    ax.set_ylabel("Factor Score", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8)
    ax.grid(axis="y", alpha=0.3, linewidth=0.6)

plt.suptitle("Factor Score Trajectories: Top Countries per Factor  (2000–2018)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("outputs/factor_trajectories.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: factor_trajectories.png")

#tracking overall scores for a handpicked set of interesting countries
benchmark = ["Sweden", "Denmark", "Germany", "Switzerland", "Portugal",
             "Korea, Rep.", "Chile", "India", "Brazil", "Nigeria"]
benchmark = [c for c in benchmark if c in scores_df["Country Name"].values]

fig, ax = plt.subplots(figsize=(13, 7))
cmap2 = plt.cm.tab10
for i, country in enumerate(benchmark):
    sub = scores_df[scores_df["Country Name"] == country].sort_values("Year")
    ax.plot(sub["Year"], sub["Composite Score"], marker="o", markersize=3,
            label=country, color=cmap2(i / len(benchmark)), linewidth=1.8)
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Composite Score (weighted)", fontsize=11)
ax.set_title("Composite Sustainability Score Trajectories — Benchmark Countries",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9, loc="upper left", framealpha=0.85)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/trajectory_benchmark.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: trajectory_benchmark.png")

print(f"DONE — {len(os.listdir('outputs'))} files saved to ./outputs/")





STEP 7 — Generating charts
  Saved: top30_leaderboard.png
  Saved: factor_loadings_heatmap.png
  Saved: communalities.png
  Saved: score_vs_trajectory.png
  Saved: metric_correlation_heatmap.png
  Saved: country_factor_heatmap.png
  Saved: factor_trajectories.png
  Saved: trajectory_benchmark.png
DONE — 20 files saved to ./outputs/
